# EDA - Taco Bueno Pricing

Análisis exploratorio de 65M+ transacciones de Taco Bueno (ene 2025 — sep 2026) para identificar descuentos que no funcionaron, oportunidades de aumentar tráfico por tienda, y el costo-beneficio de estrategias de descuento.

## 1. Resumen de Limpieza de Datos

**Tabla limpia:** `workspace.default.taco_bueno_clean_data` — 65,082,100 filas, 19 columnas (de 85.3M filas y 22 columnas en la tabla raw)

### Datos eliminados

| Qué | Cuánto | Por qué |
|---|---|---|
| Duplicados exactos | 20,189,645 (23.68%) | CSVs solapados; filas aparecían hasta 20x |
| Columnas 100% nulas | 3 (`time_card`, `menu_item_ext_ref`, `total_rounding`) | Sin información útil |
| Outliers `quantity` > 50 | 1,070 filas | Errores POS (salsas con qty=3,996) |
| Outliers `menu_price` > $50 | 1,294 filas | No-comida (donaciones, gift cards, test $11,111) |
| `transaction_type = 'Void'` | 224,280 filas | No son ventas reales; excluidas del análisis de tráfico |

### Datos mantenidos

| Qué | Por qué |
|---|---|
| `menu_price` negativo (287K filas) | Son promociones válidas ("Item Promo") |
| `pop = '3'` (3.75M, 4.4%) | Atributo POS válido, no un error |
| `total` negativo (54 filas) | Mínimo -$0.35; se analizan por separado |

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import numpy as np

# Estilo global: fondo blanco, sin grid, colores diferenciables
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': False,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'figure.figsize': (12, 5),
    'legend.frameon': False,
})

# Paleta de 12 colores altamente diferenciables
COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800',
          '#795548', '#607D8B', '#E91E63', '#3F51B5', '#009688',
          '#FFC107', '#00BCD4']

import os

CHARTS_DIR = '/Volumes/workspace/default/taco_bueno_data/charts'
os.makedirs(CHARTS_DIR, exist_ok=True)

CHART_NAMES = [
    '01_monthly_trend',
    '02_transactions_by_day_of_week',
    '03_transactions_by_hour',
    '04_top_10_stores',
    '05_store_distribution',
    '06_top_20_products',
    '07_revenue_by_category',
    '08_channel_distribution',
    '09_channel_evolution',
    '10_discount_evolution',
    '11_discount_vs_traffic'
]
_chart_idx = [0]

def save_svg(fig, name=None):
    _chart_idx[0] += 1
    name = CHART_NAMES[_chart_idx[0] - 1] if _chart_idx[0] <= len(CHART_NAMES) else f'chart_{_chart_idx[0]:02d}'
    path = f'{CHARTS_DIR}/{name}.svg'
    fig.savefig(path, format='svg', bbox_inches='tight', facecolor='white')
    display(fig)
    plt.close()

## 2. Descripción General del Dataset

In [0]:
# Estadísticas generales del dataset limpio (excluyendo Void)
display(spark.sql("""
SELECT 
  COUNT(*) as total_filas,
  COUNT(DISTINCT store_id) as num_tiendas,
  MIN(date) as fecha_min,
  MAX(date) as fecha_max,
  DATEDIFF(MAX(date), MIN(date)) as dias_con_datos,
  COUNT(DISTINCT menu_item) as num_productos,
  COUNT(DISTINCT pop) as num_canales
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
"""))

## 3. Análisis Temporal

In [0]:
# Tendencia mensual: transacciones, tiendas activas y revenue
df = spark.sql("""
SELECT 
  DATE_TRUNC('month', date) as mes,
  COUNT(*) as transacciones,
  COUNT(DISTINCT store_id) as tiendas_activas,
  ROUND(SUM(total), 0) as revenue
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY 1
ORDER BY 1
""").toPandas()
df['mes'] = pd.to_datetime(df['mes'])

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(df['mes'], df['transacciones'], color=COLORS[0], alpha=0.7, width=20, label='Transactions')
ax1.set_xlabel('Month')
ax1.set_ylabel('Transactions', color=COLORS[0])
ax1.tick_params(axis='y', labelcolor=COLORS[0])
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

ax2 = ax1.twinx()
ax2.grid(False)
ax2.plot(df['mes'], df['revenue'], color=COLORS[1], marker='o', linewidth=2, markersize=5, label='Revenue ($)')
ax2.set_ylabel('Revenue ($)', color=COLORS[1])
ax2.tick_params(axis='y', labelcolor=COLORS[1])
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))

plt.title('Monthly Trend: Transactions & Revenue')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left')
plt.tight_layout()
save_svg(fig)

In [0]:
# Transacciones por día de la semana
df_dow = spark.sql("""
SELECT 
  DAYOFWEEK(date) as dow_num,
  DATE_FORMAT(date, 'EEEE') as dia_semana,
  COUNT(*) as transacciones,
  ROUND(SUM(total), 0) as revenue
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY 1, 2
ORDER BY 1
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df_dow['dia_semana'], df_dow['transacciones'], color=COLORS[:7])
ax.set_xlabel('Day of Week')
ax.set_ylabel('Transactions')
ax.set_title('Transactions by Day of Week')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.bar_label(bars, fmt='%,.0f', fontsize=9, padding=3)
plt.tight_layout()
save_svg(fig)

# Transacciones por hora del día
df_hour = spark.sql("""
SELECT 
  HOUR(begin_time) as hora,
  COUNT(*) as transacciones
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void' AND begin_time IS NOT NULL
GROUP BY 1
ORDER BY 1
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df_hour['hora'], df_hour['transacciones'], color=COLORS[0])
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Transactions')
ax.set_title('Transactions by Hour of Day')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
save_svg(fig)

## 4. Análisis por Tienda

In [0]:
# Top 10 tiendas por volumen de transacciones
df_top = spark.sql("""
SELECT store_id, COUNT(*) as transacciones, ROUND(SUM(total), 0) as revenue
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY store_id
ORDER BY transacciones DESC
LIMIT 10
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_top['store_id'].astype(str), df_top['transacciones'], color=COLORS[:10])
ax.set_xlabel('Transactions')
ax.set_ylabel('Store ID')
ax.set_title('Top 10 Stores by Transactions')
ax.invert_yaxis()
ax.bar_label(bars, fmt='%,.0f', fontsize=9, padding=3)
plt.tight_layout()
save_svg(fig)

# Distribución de transacciones por tienda (todas las tiendas)
df_all = spark.sql("""
SELECT store_id, COUNT(*) as transacciones
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY store_id
ORDER BY transacciones DESC
""").toPandas()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(df_all)), df_all['transacciones'], color=COLORS[0])
ax.set_xlabel('Stores (sorted by volume)')
ax.set_ylabel('Transactions')
ax.set_title('Transaction Distribution by Store (119 stores)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
save_svg(fig)

## 5. Análisis de Productos

In [0]:
# Top 20 productos más vendidos por cantidad
df_prod = spark.sql("""
SELECT menu_item, COUNT(*) as transacciones, SUM(quantity) as cantidad_vendida, ROUND(SUM(total), 0) as revenue
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY menu_item
ORDER BY cantidad_vendida DESC
LIMIT 20
""").toPandas()
df_prod['menu_item'] = df_prod['menu_item'].fillna('N/A')

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(df_prod['menu_item'], df_prod['cantidad_vendida'], color=COLORS[0])
ax.set_xlabel('Quantity Sold')
ax.set_ylabel('Product')
ax.set_title('Top 20 Products by Quantity Sold')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
save_svg(fig)

# Revenue por categoría (report_group)
df_cat = spark.sql("""
SELECT report_group, COUNT(*) as transacciones, ROUND(SUM(total), 0) as revenue
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY report_group
ORDER BY revenue DESC
""").toPandas()
df_cat['report_group'] = df_cat['report_group'].fillna('N/A')

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(df_cat['report_group'], df_cat['revenue'], color=COLORS[:len(df_cat)])
ax.set_xlabel('Revenue ($)')
ax.set_ylabel('Category')
ax.set_title('Revenue by Category (report_group)')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
plt.tight_layout()
save_svg(fig)

## 6. Análisis de Canales (`pop`)

In [0]:
# Distribución de canales y ticket promedio
df_chan = spark.sql("""
SELECT 
  pop as canal,
  COUNT(*) as transacciones,
  ROUND(SUM(total), 0) as revenue,
  ROUND(AVG(total), 2) as ticket_promedio
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY 1
ORDER BY transacciones DESC
""").toPandas()

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    df_chan['transacciones'], labels=df_chan['canal'], autopct='%1.1f%%',
    colors=COLORS[:len(df_chan)], startangle=90, textprops={'fontsize': 9}
)
ax.set_title('Transaction Distribution by Channel')
plt.tight_layout()
save_svg(fig)

# Evolución temporal de canales (transacciones por mes)
df_evo = spark.sql("""
SELECT 
  DATE_TRUNC('month', date) as mes,
  pop as canal,
  COUNT(*) as transacciones
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY 1, 2
ORDER BY 1, 2
""").toPandas()
df_evo['mes'] = pd.to_datetime(df_evo['mes'])
pivot = df_evo.pivot(index='mes', columns='canal', values='transacciones').fillna(0)

fig, ax = plt.subplots(figsize=(12, 6))
for i, col in enumerate(pivot.columns):
    ax.plot(pivot.index, pivot[col], marker='o', linewidth=2, markersize=4, label=col, color=COLORS[i % len(COLORS)])
ax.set_xlabel('Month')
ax.set_ylabel('Transactions')
ax.set_title('Monthly Evolution by Channel')
ax.legend(loc='upper right', fontsize=9)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
save_svg(fig)

## 7. Análisis de Descuentos

In [0]:
# Evolución mensual: tasa de descuento, ticket con/sin descuento, monto total descontado
df = spark.sql("""
SELECT 
  DATE_TRUNC('month', date) as mes,
  COUNT(*) as transacciones,
  SUM(CASE WHEN total_discount > 0 THEN 1 ELSE 0 END) as con_descuento,
  ROUND(SUM(CASE WHEN total_discount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_descuento,
  ROUND(AVG(CASE WHEN total_discount > 0 THEN total END), 2) as ticket_con_descuento,
  ROUND(AVG(CASE WHEN total_discount = 0 THEN total END), 2) as ticket_sin_descuento,
  ROUND(SUM(total_discount), 0) as descuento_total
FROM workspace.default.taco_bueno_clean_data
WHERE transaction_type != 'Void'
GROUP BY 1
ORDER BY 1
""").toPandas()
df['mes'] = pd.to_datetime(df['mes'])

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(df['mes'], df['pct_descuento'], color=COLORS[1], marker='o', linewidth=2, markersize=5, label='Discount %')
ax1.set_xlabel('Month')
ax1.set_ylabel('Discount %', color=COLORS[1])
ax1.tick_params(axis='y', labelcolor=COLORS[1])

ax2 = ax1.twinx()
ax2.grid(False)
ax2.plot(df['mes'], df['ticket_con_descuento'], color=COLORS[0], marker='s', linewidth=2, markersize=5, label='Ticket with discount')
ax2.plot(df['mes'], df['ticket_sin_descuento'], color=COLORS[2], marker='^', linewidth=2, markersize=5, label='Ticket without discount')
ax2.set_ylabel('Avg Ticket ($)', color='#333333')
ax2.tick_params(axis='y', labelcolor='#333333')

plt.title('Discount Evolution: Rate & Avg Ticket')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper right')
plt.tight_layout()
save_svg(fig)

In [0]:
# Correlación: descuento vs tráfico (órdenes por tienda por día)
# Compara la tasa de descuento mensual con el tráfico promedio por tienda
df = spark.sql("""
WITH store_daily AS (
  SELECT 
    store_id,
    date,
    COUNT(DISTINCT check_number) as ordenes,
    COUNT(DISTINCT CASE WHEN total_discount > 0 THEN check_number END) as ordenes_con_descuento
  FROM workspace.default.taco_bueno_clean_data
  WHERE transaction_type != 'Void'
  GROUP BY store_id, date
),
monthly AS (
  SELECT 
    DATE_TRUNC('month', date) as mes,
    store_id,
    AVG(ordenes) as avg_daily_orders,
    SUM(ordenes_con_descuento) * 100.0 / SUM(ordenes) as discount_rate
  FROM store_daily
  GROUP BY 1, 2
)
SELECT 
  mes,
  ROUND(AVG(avg_daily_orders), 1) as avg_ordenes_por_tienda_dia,
  ROUND(AVG(discount_rate), 2) as pct_descuento_promedio
FROM monthly
GROUP BY 1
ORDER BY 1
""").toPandas()
df['mes'] = pd.to_datetime(df['mes'])

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(df['mes'], df['avg_ordenes_por_tienda_dia'], color=COLORS[0], marker='o', linewidth=2, markersize=5, label='Orders/store/day')
ax1.set_xlabel('Month')
ax1.set_ylabel('Avg Orders per Store per Day', color=COLORS[0])
ax1.tick_params(axis='y', labelcolor=COLORS[0])

ax2 = ax1.twinx()
ax2.grid(False)
ax2.plot(df['mes'], df['pct_descuento_promedio'], color=COLORS[1], marker='s', linewidth=2, markersize=5, label='Discount %')
ax2.set_ylabel('Avg Discount %', color=COLORS[1])
ax2.tick_params(axis='y', labelcolor=COLORS[1])

plt.title('Discount vs Traffic: More Discounts = More Orders?')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper right')
plt.tight_layout()
save_svg(fig)